# Delete Entries from a Collection Using Filters
This notebook demonstrates how to use the Weaviate filter class to delete entries from a collection. Since Weaviate collections have different configurations based on how they were made, this notebook can come in handy testing those pesky metadata typing errors. Using this notebook, you can ingest directly into your intended collection and then delete the tests as you go. We also demonstrate how to delete entries from a collection without using the filter class. This manual method can be more reliable for larger datasets. 

In [ ]:
import logging
import os
from dotenv import load_dotenv

import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter
from weaviate.classes.init import AdditionalConfig, Timeout

In [ ]:
logging.basicConfig(level=logging.INFO)

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

## Delete using Filter class

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,  # Default is 80, WCD uses 443
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,  # Default is 50051, WCD uses 443
        grpc_secure=False,
        auth_credentials=Auth.api_key(
            weaviate_api_key
        ),
        headers={"X-OpenAI-Api-Key": openai_api_key},
        additional_config=AdditionalConfig(
            timeout=Timeout(init=30, query=60, insert=150)
        ),
        skip_init_checks=True,
    )
    print("Client is live:", client.is_live())

    # Name of the class you're working with
    INDEX_NAME = "Ingestion_20250610"

    collection = client.collections.get(INDEX_NAME)
    start = collection.aggregate.over_all(total_count=True)
    print(f"Total number of objects to start in '{INDEX_NAME}': {start.total_count}")
    
    collection.data.delete_many(
        where=(
            Filter.by_property("source_key").equal("paper")
        )
    )

    end = collection.aggregate.over_all(total_count=True)
    print(f"Total number of objects remaining in '{INDEX_NAME}': {end.total_count}")
    print(f"Deleted {start.total_count - end.total_count} objects.")

except Exception as e:
    print("Error:", e)
finally:
    client.close()

## Custom delete function

In [ ]:
def custom_delete(index_name: str,
                      filter_property: str,
                      property_keys: list[str]
    ) -> None: 
        """A function to delete entries by property.
        
        Parameters
        ----------
        index_name: str
            Name of the Weaviate collection.
        filter_property: str
            The collection property to filter by (e.g. source_key).
            Properties correspond to metadata from LangChain Documents.
        property_keys: list[str]
            The specific keys to delete (e.g. ['github', 'lsst_bib'] for
            would delete every entry from those sources, if source_key
            was the filter_property).
        """
        to_delete = []

        # Access the class
        collection = client.collections.get(index_name)
        response = collection.aggregate.over_all(total_count=True)
        print(f"Total number of objects before delete '{index_name}': {response.total_count}")
        
        # Iterate through the collection
        for item in collection.iterator():
            properties = getattr(item, 'properties', {})
            key = properties.get(filter_property, None)
        
            # Check if key matches filter
            if key in property_keys:
                uuid = item.uuid
                to_delete.append(uuid)
                 
        print(f"Found {len(to_delete)} objects to delete.")
        
        # Proceed to delete the unwanted objects
        print("Deleting")
        for uuid in to_delete:
            collection.data.delete_by_id(uuid)
        response = collection.aggregate.over_all(total_count=True)
        print(f"Total number of objects after delete '{index_name}': {response.total_count}")

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,  # Default is 80, WCD uses 443
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,  # Default is 50051, WCD uses 443
        grpc_secure=False,
        auth_credentials=Auth.api_key(
            weaviate_api_key
        ),  # The API key to use for authentication
        headers={"X-OpenAI-Api-Key": openai_api_key},
        skip_init_checks=True,
    )
    print("Client is live:", client.is_live())

    custom_delete("LangChain_9787ec4b92d3438a8de3ff04ead7ead6", ["paper"])

except Exception as e:
    print("Error:", e)
finally:
    client.close()